In [1]:
import folium
import pandas as pd
# folium은 .save()로 HTML 지도 파일을 만들고, 브라우저로 열어 확인합니다(Jupyter는 그대로 표시)

In [2]:
import folium
# 부산 좌표 중심, zoom_start 클수록 확대
m = folium.Map(location=[35.156023, 129.059407], zoom_start=15)
m.save("map_basic.html")    # 브라우저로 열어 확인

In [5]:
import folium
import pandas as pd

# 샘플 데이터 열: store, address, phone, address2, _GC_TYPE, _CLEANADDR, _X(경도), _Y(위도)
df = pd.read_csv("./CB_geo_shp.csv", encoding="utf-8")

# 좌표 열을 숫자로 변환하고, 좌표가 없는 행은 제거
df["_Y"] = pd.to_numeric(df["_Y"], errors="coerce")  # 위도
df["_X"] = pd.to_numeric(df["_X"], errors="coerce")  # 경도
df = df.dropna(subset=["_Y", "_X"])

# 지도 중심을 데이터 평균 좌표로 설정 (하드코딩 대신 데이터 기반)
center = [df["_Y"].mean(), df["_X"].mean()]
m = folium.Map(location=center, zoom_start=12)

for i, store in df.iterrows():  # 한 행씩 반복
    folium.Marker(
        location=[store["_Y"], store["_X"]],  # [위도, 경도] 순서
        popup=folium.Popup(
            f"<b>{store['store']}</b><br>{store['address']}<br>{store['phone']}",
            max_width=250,
        ),
        tooltip=store["store"],  # 마우스 오버 시 매장명 표시
    ).add_to(m)

m.save("map_markers.html")
print(f"마커 {len(df)}개 생성 완료 -> map_markers.html")

마커 243개 생성 완료 -> map_markers.html


In [6]:
# 주소 문자열을 공백으로 잘라 1차(시도)·2차(군구) 지역명 추출
addr = []
for a in df["address"]:
    parts = str(a).split()
    addr.append({"시도": parts[0], "군구": parts[1]})
addr = pd.DataFrame(addr)

# 지역별 개수 집계 + 인구수 테이블과 병합(inner join)
addr["count"] = 1
by_region = addr.groupby(["시도", "군구"])["count"].sum().reset_index()
# population: 지역별 인구수 데이터프레임(세트 인덱스 맞춰 merge)
# merged = pd.merge(by_region, population, on=["시도","군구"], how="inner")

# 인구 대비 비율(10만명당) = (개수 / 인구수) * 100000
# merged["ratio"] = merged["count"] / merged["인구수"] * 100000

In [7]:
import matplotlib.pyplot as plt
import platform
plt.rc("font", family="AppleGothic" if platform.system()=="Darwin" else "Malgun Gothic")
plt.rcParams["axes.unicode_minus"] = False

# ratio 기준 내림차순 정렬 후 막대그래프(예시)
# top = merged.sort_values("ratio", ascending=False).head(10)
# plt.barh(top["군구"], top["ratio"]); plt.title("인구 대비 의료기관 비율 상위 10"); plt.show()